In [1]:
from pathlib import Path
import gcamreader
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [5]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250715_2"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [6]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [7]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [8]:
i = 159
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

transport service output by tech


,Units,scenario,region,sector,subsector,technology,Year,value
0,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,BEV,2035,3.474780e-03
1,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Hydrogen,2035,1.010580e-03
2,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1975,6.166440e+02
3,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1990,4.066310e+03
4,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,2005,6.792510e+03
...,...,...,...,...,...,...,...,...
557,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2015,3.207670e+06
558,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2020,3.168000e+06
559,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2025,3.032060e+06
560,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2030,2.359320e+06


In [9]:
df['ZEV'] = df['technology'].apply(lambda tech: 1 if tech in ['BEV', 'FCEV'] else 0)
df

,Units,scenario,region,sector,subsector,technology,Year,value,ZEV
0,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,BEV,2035,3.474780e-03,1
1,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Hydrogen,2035,1.010580e-03,0
2,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1975,6.166440e+02,0
3,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1990,4.066310e+03,0
4,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,2005,6.792510e+03,0
...,...,...,...,...,...,...,...,...,...
557,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2015,3.207670e+06,0
558,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2020,3.168000e+06,0
559,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2025,3.032060e+06,0
560,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2030,2.359320e+06,0


In [10]:
df[(df['ZEV'] == 1)]['subsector'].unique()

array(['International Aviation', 'Domestic Aviation', 'Bus', '2W and 3W',
       'Car', 'Large Car and Truck', 'Domestic Ship', 'Freight Rail',
       'Medium truck', 'International Ship'], dtype=object)

In [11]:
lstServiceSectors = ['Bus', 'Car', 'Large Car and Truck', 'Medium truck']

In [12]:
dfService = df[(df['subsector'].isin(lstServiceSectors))].copy()
serServiceZev = dfService[(dfService['Year'] >= 2020) & (dfService['ZEV'] == 1)].groupby(['scenario', 'Year', 'Units'])['value'].sum()
serServiceTot = dfService[(dfService['Year'] >= 2020)].groupby(['scenario', 'Year', 'Units'])['value'].sum()
dfServiceRatio = (serServiceZev / serServiceTot).reset_index()
dfServiceRatio

,scenario,Year,Units,value
0,Current-Policy,2020,million pass-km,0.004787
1,Current-Policy,2020,million ton-km,0.010923
2,Current-Policy,2025,million pass-km,0.096420
3,Current-Policy,2025,million ton-km,0.051348
4,Current-Policy,2030,million pass-km,0.160928
5,Current-Policy,2030,million ton-km,0.085067
6,Current-Policy,2035,million pass-km,0.223149
7,Current-Policy,2035,million ton-km,0.130336
8,Enhanced-Ambition,2020,million pass-km,0.004787
9,Enhanced-Ambition,2020,million ton-km,0.010923


In [13]:
dfServiceZev = serServiceZev.reset_index()
dfServiceZev['value'] /= 1000

In [14]:
i = 161
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

transport service output by tech (new)


,Units,scenario,region,sector,subsector,technology,Year,value
0,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,BEV,2035,3.474780e-03
1,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Hydrogen,2035,1.010580e-03
2,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1975,6.166440e+02
3,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1990,4.066310e+03
4,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,2005,6.792510e+03
...,...,...,...,...,...,...,...,...
557,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2015,3.207670e+06
558,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2020,3.168000e+06
559,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2025,3.032060e+06
560,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2030,2.359320e+06


In [15]:
df['ZEV'] = df['technology'].apply(lambda tech: 1 if tech in ['BEV', 'FCEV'] else 0)
df

,Units,scenario,region,sector,subsector,technology,Year,value,ZEV
0,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,BEV,2035,3.474780e-03,1
1,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Hydrogen,2035,1.010580e-03,0
2,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1975,6.166440e+02,0
3,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,1990,4.066310e+03,0
4,million pass-km,Current-Policy,South Korea,trn_aviation_intl,International Aviation,Liquids,2005,6.792510e+03,0
...,...,...,...,...,...,...,...,...,...
557,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2015,3.207670e+06,0
558,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2020,3.168000e+06,0
559,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2025,3.032060e+06,0
560,million ton-km,Enhanced-Ambition,South Korea,trn_shipping_intl,International Ship,Liquids,2030,2.359320e+06,0


In [16]:
df[(df['scenario'] == scenarios[1]) & (df['Year'] >= 2025) & (df['Units'] == 'million ton-km') & (df['subsector'] == 'Medium truck')]

,Units,scenario,region,sector,subsector,technology,Year,value,ZEV
528,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,BEV,2025,1456.81,1
529,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,BEV,2030,5133.56,1
530,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,BEV,2035,8323.31,1
532,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,FCEV,2025,7468.44,1
533,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,FCEV,2030,13887.90,1
534,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,FCEV,2035,17118.00,1
536,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,Hybrid Liquids,2025,27778.90,0
537,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,Hybrid Liquids,2030,34841.30,0
538,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,Hybrid Liquids,2035,33063.10,0
544,million ton-km,Enhanced-Ambition,South Korea,trn_freight_road,Medium truck,Liquids,2025,44034.50,0


In [17]:
dfService = df[(df['subsector'].isin(lstServiceSectors))].copy()
serServiceZev = dfService[(dfService['Year'] >= 2020) & (dfService['ZEV'] == 1)].groupby(['scenario', 'Year', 'Units'])['value'].sum()
serServiceTot = dfService[(dfService['Year'] >= 2020)].groupby(['scenario', 'Year', 'Units'])['value'].sum()
dfServiceRatio = (serServiceZev / serServiceTot).reset_index()
dfServiceRatio['value'] *= 100

In [18]:
dfServiceRatio

,scenario,Year,Units,value
0,Current-Policy,2020,million pass-km,0.667830
1,Current-Policy,2020,million ton-km,3.136751
2,Current-Policy,2025,million pass-km,12.675829
3,Current-Policy,2025,million ton-km,11.074479
4,Current-Policy,2030,million pass-km,18.720573
5,Current-Policy,2030,million ton-km,11.199968
6,Current-Policy,2035,million pass-km,24.260166
7,Current-Policy,2035,million ton-km,16.920237
8,Enhanced-Ambition,2020,million pass-km,0.667830
9,Enhanced-Ambition,2020,million ton-km,3.136751


In [29]:
# Assign consistent colors for scenarios
color_map = {
    scenarios[0]: '#00CC96',   # blue
    scenarios[1]: '#AB63FA'
}


# Create subplot layout with secondary_y enabled in the first plot
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Passenger", "Freight"],
    specs=[[{"secondary_y": True}, {"secondary_y": True}]]
)

# First y-axis: Total passenger km
dfServiceRatioPass = dfServiceRatio[dfServiceRatio['Units'] == 'million pass-km']
for scenario in scenarios:
    scenario_data = dfServiceRatioPass[dfServiceRatioPass['scenario'] == scenario]
    fig.add_trace(
        go.Scatter(
            x=scenario_data['Year'],
            y=scenario_data['value'],
            name=f"New Sales in {scenario} in scenario (%)",
            mode='lines+markers',
            line=dict(color=color_map[scenario]),
        ),
        row=1,
        col=1,
        secondary_y=False
    )

# Second y-axis (right): ZEV passenger km
dfServiceZevPass = dfServiceZev[dfServiceZev['Units'] == 'million pass-km']
for scenario in scenarios:
    scenario_data = dfServiceZevPass[dfServiceZevPass['scenario'] == scenario]
    fig.add_trace(
        go.Bar(
            x=scenario_data['Year'],
            y=scenario_data['value'],
            name=f"Total Service in {scenario} scenario",
            marker_color=color_map[scenario],
            opacity=0.5
        ),
        row=1,
        col=1,
        secondary_y=True
    )

# First y-axis: Total passenger km
dfServiceRatioFreight = dfServiceRatio[dfServiceRatio['Units'] == 'million ton-km']
for scenario in scenarios:
    scenario_data = dfServiceRatioFreight[dfServiceRatioFreight['scenario'] == scenario]
    fig.add_trace(
        go.Scatter(
            x=scenario_data['Year'],
            y=scenario_data['value'],
            name=f"New Sales in {scenario} in scenario (%)",
            mode='lines+markers',
            line=dict(color=color_map[scenario]),
            showlegend=False
        ),
        row=1,
        col=2,
        secondary_y=False
    )

# Second y-axis (right): ZEV passenger km
dfServiceZevFreight = dfServiceZev[dfServiceZev['Units'] == 'million ton-km']
for scenario in scenarios:
    scenario_data = dfServiceZevFreight[dfServiceZevFreight['scenario'] == scenario]
    fig.add_trace(
        go.Bar(
            x=scenario_data['Year'],
            y=scenario_data['value'],
            name=f"Total Service in {scenario} scenario",
            marker_color=color_map[scenario],
            opacity=0.5,
            showlegend=False
        ),
        row=1,
        col=2,
        secondary_y=True
    )

# Apply layout formatting again
fig.update_layout(
    height=500,
    width=800,
    # title_text="<b>Projected ZEV Penetrations of Korea through 2035</b>",
    # title_font_size=18,
    # title_x=0.5,
    barmode='group',
    yaxis_title='Million pass–km/ton–km',
    xaxis_title=None,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(bgcolor='white'),
)

# Subplot (1,1)
fig.update_yaxes(title_text="New Sales Share (%)", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="billion pass-km", secondary_y=True, row=1, col=1)

# Subplot (1,2)
fig.update_yaxes(title_text="New Sales Share (%)", secondary_y=False, row=1, col=2)
fig.update_yaxes(title_text="billion ton-km", secondary_y=True, row=1, col=2)

fig.update_yaxes(tickformat=',d', row=1, col=1)
fig.update_yaxes(tickformat=',d', row=1, col=2)

fig.update_layout(
    legend=dict(
        x=0.5,
        y=-0.2,
        xanchor='center',
        yanchor='top',
        orientation='h',  # horizontal layout
        bgcolor='white',
        bordercolor='lightgray',
        borderwidth=1,
        font=dict(size=18),
    )
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
# fig.update_layout(title=dict(font=dict(size=25)))

fig.show()